# Risk models: crypto 24h risk snapshot

## What
Latest cryptocurrency prices and 24-hour percent changes. Coins above the 70th percentile of |24h change| are labeled `high_vol`.

## Why this model
`GET /api/v1/data/crypto/prices` is a **cross-section**, not a price history (each `coin_id` repeats with the same latest stamp). A rolling time-series volatility model would be fake here. Cross-sectional |24h change| is the risk measure the payload actually supports.

## How to rerun
Needs only `FINUTIES_API_KEY` in `notebooks/.env`. Run top to bottom.

**Endpoint (verified 200):** `GET /api/v1/data/crypto/prices`

**Columns used:** `coin_id`, `price`, `price_change_24h`, `timestamp`, `vs_currency`

The live schema uses `price` — not `price_usd`.

In [1]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv


def resolve_notebooks_env(start_dir: Path) -> Path:
    current = start_dir.resolve()
    for candidate_root in [current, *current.parents]:
        if candidate_root.name == "notebooks":
            env_path = candidate_root / ".env"
            if env_path.exists():
                return env_path
        nested_env = candidate_root / "notebooks" / ".env"
        if nested_env.exists():
            return nested_env
    raise FileNotFoundError(
        "Missing notebooks/.env. Copy notebooks/.env.example and set FINUTIES_API_KEY. "
        "A sandbox key is POST https://data.finuties.com/api/v1/auth/sandbox"
    )


def require_frame(df: pd.DataFrame, required: list[str], min_rows: int = 1) -> None:
    if df.empty:
        raise AssertionError("Expected a non-empty frame from the FinUties API.")
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise AssertionError(f"Missing required columns {missing}. Got {list(df.columns)}")
    if len(df) < min_rows:
        raise AssertionError(f"Expected at least {min_rows} rows, got {len(df)}")


def require_finite(series: pd.Series, name: str) -> None:
    numeric = pd.to_numeric(series, errors="coerce")
    valid = numeric.dropna()
    if valid.empty:
        raise AssertionError(f"{name} has no numeric values")
    if not np.isfinite(valid.to_numpy()).all():
        raise AssertionError(f"{name} contains non-finite values")


load_dotenv(resolve_notebooks_env(Path.cwd()))
API_ORIGIN = os.getenv("FINUTIES_API_ORIGIN", "https://data.finuties.com").rstrip("/")
API_KEY = os.getenv("FINUTIES_API_KEY", "").strip()
if not API_KEY:
    raise ValueError(
        "Missing FINUTIES_API_KEY. Copy notebooks/.env.example to notebooks/.env "
        "and set a key from POST /api/v1/auth/sandbox"
    )

HEADERS = {"Authorization": f"Bearer {API_KEY}"}
TIMEOUT_SECONDS = 45


def finuties_get(endpoint: str, params: dict | None = None):
    response = requests.get(
        f"{API_ORIGIN}{endpoint}",
        headers=HEADERS,
        params=params or {},
        timeout=TIMEOUT_SECONDS,
    )
    response.raise_for_status()
    return response.json()


def normalize_rows(payload) -> list[dict]:
    if isinstance(payload, list):
        return [row for row in payload if isinstance(row, dict)]
    if isinstance(payload, dict):
        for key in ("items", "data", "rows", "results"):
            rows = payload.get(key)
            if isinstance(rows, list):
                return [row for row in rows if isinstance(row, dict)]
    return []


In [2]:
ENDPOINT = "/api/v1/data/crypto/prices"
payload = finuties_get(ENDPOINT, {"limit": 50})
df = pd.DataFrame(normalize_rows(payload))
require_frame(df, ["coin_id", "price", "price_change_24h", "timestamp"], min_rows=5)
assert "price_usd" not in df.columns

df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["price_change_24h"] = pd.to_numeric(df["price_change_24h"], errors="coerce")
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df.dropna(subset=["coin_id", "price", "price_change_24h"]).copy()
df = df.sort_values("timestamp").drop_duplicates("coin_id", keep="last")
require_frame(df, ["coin_id", "price", "price_change_24h"], min_rows=5)
require_finite(df["price"], "price")
require_finite(df["price_change_24h"], "price_change_24h")

df["abs_change_24h"] = df["price_change_24h"].abs()
threshold = float(df["abs_change_24h"].quantile(0.70))
assert np.isfinite(threshold)
df["regime"] = np.where(df["abs_change_24h"] >= threshold, "high_vol", "low_vol")
df = df.sort_values("abs_change_24h", ascending=False).reset_index(drop=True)

print(f"Coins: {len(df)}  |24h| 70th percentile: {threshold:.3f}%")
print(f"vs_currency: {df['vs_currency'].dropna().iloc[0] if 'vs_currency' in df.columns else 'usd'}")
df[["coin_id", "price", "price_change_24h", "regime", "timestamp"]].head(12)

Coins: 15  |24h| 70th percentile: 0.497%
vs_currency: usd


,coin_id,price,price_change_24h,regime,timestamp
0,polkadot,0.746405,-1.784400,high_vol,2026-08-18 20:14:15.475034+00:00
1,solana,77.050003,1.726140,high_vol,2026-08-18 20:08:44.958570+00:00
2,tron,0.332954,0.574494,high_vol,2026-08-18 20:13:36.220072+00:00
3,bitcoin,64606.000000,0.540332,high_vol,2026-08-18 19:59:48.044086+00:00
4,shiba-inu,0.000004,-0.499234,high_vol,2026-08-18 19:52:06.813866+00:00
5,wrapped-bitcoin,64650.000000,0.487579,low_vol,2026-08-18 19:52:25.084256+00:00
6,binancecoin,602.650024,-0.332959,low_vol,2026-08-18 20:05:49.623524+00:00
7,chainlink,9.500000,0.321331,low_vol,2026-08-18 19:51:12.277396+00:00
8,ethereum,1911.949951,0.314407,low_vol,2026-08-18 20:03:16.122035+00:00
9,cardano,0.173273,-0.196325,low_vol,2026-08-18 20:10:17.262463+00:00


## Charts

Left: latest `price` in USD. Right: API `price_change_24h` in percent. Red bars are coins at or above the sample 70th percentile of absolute 24h change.

In [3]:
plot_df = df.head(15)
labels = plot_df["coin_id"].astype(str)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].bar(labels, plot_df["price"], color="#1f77b4")
axes[0].set_title("Latest crypto price")
axes[0].set_xlabel("Coin id")
axes[0].set_ylabel("Price (USD)")
axes[0].tick_params(axis="x", labelrotation=45)

colors = np.where(plot_df["regime"] == "high_vol", "#d62728", "#1f77b4")
axes[1].bar(labels, plot_df["price_change_24h"], color=colors)
axes[1].axhline(0, color="#444444", linewidth=1)
axes[1].set_title("24h change (red = high_vol)")
axes[1].set_xlabel("Coin id")
axes[1].set_ylabel("24h change (%)")
axes[1].tick_params(axis="x", labelrotation=45)

for ax in axes:
    for label in ax.get_xticklabels():
        label.set_horizontalalignment("right")
plt.tight_layout()
plt.show()

## Caveats

- Snapshot only. You cannot estimate GARCH or 30-day realized vol from this endpoint.
- `price_change_24h` is already a percent from the API; it is not computed from a history we fetched.
- Duplicate `coin_id` rows were collapsed to the latest `timestamp`.
- Thresholds are sample-relative. Not investment advice.